# Teks + kolom tabel: GBDT di atas fitur teks dan kolom angka/kategori
**Kapan dipakai:** tabel yang punya kolom teks panjang (ulasan, deskripsi, judul, isian bebas)
*dan* kolom angka/kategori. Pola final 2024: isian teks bebas ikut menentukan hasil. Teks
diubah jadi angka (statistik teks + TF-IDF → SVD), digabung dengan kolom tabel, lalu satu
model pohon (LightGBM) belajar semuanya.

**Langkah:** baca → temuan data + cek bocor per kolom → fitur → CV 5 lipatan (OOF) → ablasi
(seberapa berguna teks?) → simpan OOF → `submission.csv` + validator.

**Data demo:** ulasan baju (Women's E-Commerce Clothing Reviews, CC0). Target `rating` 1–5,
teks `title` + `review_text`, kolom lain: `age`, `positive_feedback_count`, `clothing_id`,
divisi/departemen/kelas.

In [ ]:
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

## Konfigurasi
Yang diganti saat lomba hanya sel ini (baris `# GANTI saat lomba`). Kolom angka dikenali
otomatis: semua kolom selain id, target, teks, kategori, dan kolom yang dibuang.

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/teks_tabular"
OOF_FOLDER = "../../../outputs/text/oof"

DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample dari panitia
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                                # GANTI saat lomba
TARGET = "rating"                                            # GANTI saat lomba
SUB_COL = "rating"                                           # GANTI saat lomba: kolom label sample
TEXT_COLS = ["title", "review_text"]                         # GANTI saat lomba (boleh lebih dari 1)
CAT_COLS = ["clothing_id", "division_name", "department_name", "class_name"]   # GANTI saat lomba
DROP_COLS = ["recommended_ind"]                              # GANTI saat lomba: hasil cek bocor

MODEL_NAME = "lgbm_teks_tabular"   # nama file OOF: oof_<MODEL_NAME>.npy, test_<MODEL_NAME>.npy
N_FOLDS = 5
SEED = 42
SVD_DIM = 64               # jumlah kolom SVD per kolom teks
EMBEDDING_MODEL = "../../../pretrained/minilm_en"   # alt vektor kalimat ("Rakit fitur")
# EMBEDDING_MODEL = "../../../pretrained/minilm"      # multibahasa
# EMBEDDING_MODEL = "../../../pretrained/e5_base"     # multibahasa kuat, TEXT_PREFIX "query: "
TEXT_PREFIX = ""
LEAK_AUC = 0.97            # AUC satu kolom saja di atas ini dicurigai bocor

## Baca data
Teks kosong → `""`, kategori kosong → `"kosong"`; angka kosong dibiarkan (LightGBM mengerti
NaN). Tidak ada baris yang dibuang, supaya urutan OOF sama dengan urutan train.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
MISSING = train.isna().sum()
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in CAT_COLS:
    train[col] = train[col].fillna("kosong").astype(str)
    test[col] = test[col].fillna("kosong").astype(str)

skip = [ID_COL, TARGET] + TEXT_COLS + CAT_COLS + DROP_COLS
NUM_COLS = []
for col in train.columns:
    if col not in skip:
        NUM_COLS.append(col)
CLASSES = np.unique(train[TARGET])
y_idx = np.searchsorted(CLASSES, train[TARGET].values)   # kelas → 0..K-1 (wajib untuk XGBoost)
print("train:", train.shape, "| test:", test.shape, "| kelas:", CLASSES)
print("kolom angka:", NUM_COLS)
not_number = train[NUM_COLS].select_dtypes(exclude="number")
print("bukan angka (pindah ke CAT_COLS/DROP_COLS):", list(not_number))
train.head(3)

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya sample_submission. Saat lomba `SAMPLE_CSV` = file dari panitia.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = CLASSES[0]
demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Temuan data
Sebaran kelas, nilai kosong, panjang teks, dan teks kembar. Teks kembar dengan label beda =
label berisik. Teks test yang persis ada di train membuat skor CV terlalu optimis (lipatan
tidak memisahkan kembar).

In [ ]:
def duplicate_report(col):
    filled = train[train[col] != ""]
    labels_per_text = filled.groupby(col)[TARGET].nunique()
    in_train = test[col].isin(train[col]) & (test[col] != "")
    print(f"  kembar di train: {filled[col].duplicated().sum()} | "
          f"kembar beda label: {(labels_per_text > 1).sum()} | test ada di train: {in_train.sum()}")


print(train[TARGET].value_counts().sort_index())
print("kelas terkecil:", train[TARGET].value_counts().min(), "(harus >= N_FOLDS)")
print("nilai kosong sebelum diisi:", MISSING[MISSING > 0].to_dict())
for col in TEXT_COLS:
    words = train[col].str.split().str.len()
    print(col, "| teks kosong:", (words == 0).sum(), "| kata p50/p95/p99:",
          words.quantile([0.5, 0.95, 0.99]).values)
    duplicate_report(col)

## Cek bocor per kolom
Tiap kolom dicoba **sendirian** (pohon kecil untuk angka/kategori, TF-IDF + LogReg untuk
teks), AUC lewat CV. `urutan_baris` menangkap train yang terurut menurut label. AUC mendekati
1,0 = kolom itu hampir salinan target atau dibuat setelah kejadian → masukkan ke `DROP_COLS`.
AUC tinggi tapi wajar (mis. teks untuk sentimen) bukan bocor; yang dicurigai: kolom tabel
dengan AUC sangat tinggi.

In [ ]:
SCORING = "roc_auc_ovr"
if len(CLASSES) == 2:
    SCORING = "roc_auc"


def column_auc(table, col, is_text):
    if is_text:
        tfidf = TfidfVectorizer(max_features=3000, min_df=2)
        model = make_pipeline(tfidf, LogisticRegression(max_iter=300))
        values = table[col]
    else:
        model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=SEED)
        values = table[[col]].copy()
        if not pd.api.types.is_numeric_dtype(values[col]):
            values[col] = values[col].astype("category").cat.codes
        values = values.fillna(-999)
    return cross_val_score(model, values, y_idx, cv=5, scoring=SCORING).mean()


scan_table = train.copy()
scan_table["urutan_baris"] = np.arange(len(train))
rows = []
for col in scan_table.columns:
    if col != TARGET:
        auc = column_auc(scan_table, col, col in TEXT_COLS)
        rows.append({"kolom": col, "auc_sendiri": round(auc, 4)})
leak = pd.DataFrame(rows).sort_values("auc_sendiri", ascending=False)
print(leak.to_string(index=False))
suspect = leak[leak["auc_sendiri"] >= LEAK_AUC]
print("CURIGA BOCOR (cek maknanya, lalu DROP_COLS):", list(suspect["kolom"]))

## Fitur 1: statistik teks
Panjang, jumlah kata, huruf besar, angka, tanda baca, tanda seru/tanya, kata unik, apakah
kosong. Pohon bisa memakai ini langsung (ulasan panjang sering beda dari yang pendek).

In [ ]:
def count_unique(words):
    return len(set(words))


def text_stats(table, col):
    text = table[col].fillna("")
    words = text.str.split()
    out = pd.DataFrame(index=table.index)
    out[f"{col}_n_char"] = text.str.len()
    out[f"{col}_n_word"] = words.str.len()
    out[f"{col}_n_unique"] = words.apply(count_unique)
    out[f"{col}_n_upper"] = text.str.count(r"[A-Z]")
    out[f"{col}_n_digit"] = text.str.count(r"\d")
    out[f"{col}_n_punct"] = text.str.count(r"[^\w\s]")
    out[f"{col}_n_excl"] = text.str.count("!")
    out[f"{col}_n_question"] = text.str.count(r"\?")
    out[f"{col}_upper_ratio"] = out[f"{col}_n_upper"] / (out[f"{col}_n_char"] + 1)
    out[f"{col}_empty"] = (out[f"{col}_n_char"] == 0).astype(int)
    return out


summary = text_stats(train, TEXT_COLS[0]).describe()
print(summary.loc[["mean", "50%", "max"]].round(2).T)

## Fitur 2: TF-IDF → TruncatedSVD
TF-IDF kata 1–2 gram dipadatkan jadi `SVD_DIM` kolom angka (makna kasar teks). TF-IDF dan SVD
dilatih di teks train + test: tidak memakai label, jadi tidak bocor, dan kosakata test ikut
terwakili. Alternatif: vektor kalimat pretrained (sel berikutnya, ber-`#` di sel "Rakit
fitur").

In [ ]:
def tfidf_svd(text_all, name, dim):
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True)
    matrix = tfidf.fit_transform(text_all)
    dim = min(dim, matrix.shape[1] - 1)
    vectors = TruncatedSVD(n_components=dim, random_state=SEED).fit_transform(matrix)
    names = []
    for i in range(dim):
        names.append(f"{name}_svd{i}")
    return pd.DataFrame(vectors.astype("float32"), columns=names, index=text_all.index)


def embed_svd(text_all, name, dim, folder, prefix=""):
    # alternatif: vektor kalimat dari pretrained/, lalu dipadatkan SVD agar jumlah kolom tetap kecil
    encoder = SentenceTransformer(folder, device="cpu")
    vectors = encoder.encode(list(prefix + text_all), batch_size=64, show_progress_bar=False)
    dim = min(dim, vectors.shape[1] - 1)
    small = TruncatedSVD(n_components=dim, random_state=SEED).fit_transform(vectors)
    names = []
    for i in range(dim):
        names.append(f"{name}_emb{i}")
    return pd.DataFrame(small.astype("float32"), columns=names, index=text_all.index)


demo_svd = tfidf_svd(train[TEXT_COLS[0]], TEXT_COLS[0], 8)   # uji cepat satu kolom
print("contoh SVD:", demo_svd.shape)

## Rakit fitur
Kolom tabel + statistik teks + SVD teks + kategori (`category` dibangun dari train + test,
supaya kategori yang hanya ada di test tidak membuat error). `GROUPS` mencatat kolom tiap
kelompok untuk ablasi.

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
GROUPS = {"tabel": list(NUM_COLS) + list(CAT_COLS), "statistik": [], "teks": []}
blocks = [all_data[NUM_COLS]]
for col in TEXT_COLS:
    stats = text_stats(all_data, col)
    svd = tfidf_svd(all_data[col], col, SVD_DIM)
    # svd = embed_svd(all_data[col], col, SVD_DIM, EMBEDDING_MODEL, TEXT_PREFIX)   # alt vektor
    GROUPS["statistik"] = GROUPS["statistik"] + list(stats.columns)
    GROUPS["teks"] = GROUPS["teks"] + list(svd.columns)
    blocks = blocks + [stats, svd]
for col in CAT_COLS:
    blocks.append(all_data[col].astype("category"))
features = pd.concat(blocks, axis=1)
X_train = features.iloc[: len(train)].reset_index(drop=True)
X_test = features.iloc[len(train):].reset_index(drop=True)
print("fitur:", X_train.shape)
for group in GROUPS:
    print("  ", group, len(GROUPS[group]), "kolom")

## Model
Pilih SATU: LightGBM (aktif), atau hapus `#` pada alternatif dan beri `#` pada LightGBM.
Semuanya memakai `X_train` yang sama (kecuali CatBoost `text_features`, lihat sel berikut).

In [ ]:
model = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, colsample_bytree=0.5,
                       subsample=0.8, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1)
# model = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, colsample_bytree=0.5,
#                        subsample=0.8, subsample_freq=1, class_weight="balanced",
#                        random_state=SEED, n_jobs=-1, verbose=-1)     # kelas timpang berat
# model = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
#                       colsample_bytree=0.5, tree_method="hist", enable_categorical=True,
#                       random_state=SEED, n_jobs=-1)                  # pembanding GBDT lain
print(type(model).__name__)

### Alternatif: CatBoost dengan teks mentah (`text_features`)
CatBoost membuat fitur teks sendiri (BoW + Naive Bayes di dalam), jadi tidak butuh SVD. Hapus
`#` pada tiga pernyataan terakhir (menimpa `X_train`, `X_test`, `model` dari sel sebelumnya).
Lebih lambat: ukur dulu dengan satu lipatan. Untuk kategori banyak nilai, CatBoost sering
lebih stabil.

In [ ]:
def raw_frame(table):
    parts = [table[NUM_COLS + CAT_COLS + TEXT_COLS]]
    for col in TEXT_COLS:
        parts.append(text_stats(table, col))
    return pd.concat(parts, axis=1)


print("bentuk fitur CatBoost:", raw_frame(train).shape)
# X_train = raw_frame(train)
# X_test = raw_frame(test)
# model = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, text_features=TEXT_COLS,
#                            cat_features=CAT_COLS, random_seed=SEED, verbose=0, thread_count=-1,
#                            allow_writing_files=False)

## CV 5 lipatan → OOF
`StratifiedKFold(N_FOLDS, shuffle, SEED)` pada urutan baris train apa adanya (format bersama
seluruh contoh teks). OOF = peluang tiap kelas untuk baris yang tidak dipakai melatih; test =
rata-rata model per lipatan.

In [ ]:
def run_cv(model, X, y_idx, X_test, verbose=True):
    oof = np.zeros((len(X), len(CLASSES)), dtype="float32")
    test_pred = np.zeros((len(X_test), len(CLASSES)), dtype="float32")
    folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (idx_fit, idx_val) in enumerate(folds.split(X, y_idx)):
        fitted = copy.deepcopy(model)
        fitted.fit(X.iloc[idx_fit], y_idx[idx_fit])
        oof[idx_val] = fitted.predict_proba(X.iloc[idx_val])
        test_pred += fitted.predict_proba(X_test) / N_FOLDS
        score = f1_score(y_idx[idx_val], oof[idx_val].argmax(axis=1), average="macro")
        if verbose:
            seconds = round(time.time() - START)
            print("lipatan", fold, "F1 makro", round(score, 4), "| detik", seconds)
    return oof, test_pred


START = time.time()
oof, test_pred = run_cv(model, X_train, y_idx, X_test)
SECONDS = round(time.time() - START)

## Evaluasi OOF
Metrik lomba belum tentu F1 makro: ganti `SCORE` bila perlu (mis. akurasi, QWK untuk ordinal).

In [ ]:
oof_labels = oof.argmax(axis=1)
SCORE = f1_score(y_idx, oof_labels, average="macro")
print("F1 makro OOF:", round(SCORE, 4), "| akurasi:", round(accuracy_score(y_idx, oof_labels), 4),
      "| log loss:", round(log_loss(y_idx, oof), 4), "| detik:", SECONDS)
print(classification_report(y_idx, oof_labels, target_names=CLASSES.astype(str)))
ConfusionMatrixDisplay.from_predictions(y_idx, oof_labels, display_labels=CLASSES.astype(str))
plt.show()

## Kesalahan menurut panjang teks
Skor per lima kelompok jumlah kata (teks gabungan). Kalau teks pendek jauh lebih buruk,
tambahkan fitur lain atau tangani terpisah.

In [ ]:
total_words = pd.Series(0, index=train.index)
for col in TEXT_COLS:
    total_words = total_words + train[col].str.split().str.len()
bucket = pd.qcut(total_words, 5, duplicates="drop")
report = pd.DataFrame({"benar": oof_labels == y_idx, "kelompok": bucket})
by_bucket = report.groupby("kelompok", observed=True)["benar"].agg(["mean", "size"])
print(by_bucket.round(3))

## Ablasi: seberapa berguna teks?
Model LightGBM yang sama, tiga set fitur: kolom tabel saja → + statistik teks → + SVD teks.
Selisihnya = sumbangan teks, bahan slide metode. (Ablasi memakai `features` LightGBM, bukan
CatBoost.)

In [ ]:
ablation_model = LGBMClassifier(n_estimators=200, learning_rate=0.1, colsample_bytree=0.5,
                                subsample=0.8, subsample_freq=1, random_state=SEED, verbose=-1)
lgbm_train = features.iloc[: len(train)].reset_index(drop=True)
lgbm_test = features.iloc[len(train):].reset_index(drop=True)
columns = list(GROUPS["tabel"])
rows = []
for group in ["tabel", "statistik", "teks"]:
    if group != "tabel":
        columns = columns + GROUPS[group]
    ablation_oof, _ = run_cv(ablation_model, lgbm_train[columns], y_idx, lgbm_test[columns], False)
    score = f1_score(y_idx, ablation_oof.argmax(axis=1), average="macro")
    rows.append({"fitur sampai": group, "kolom": len(columns), "F1 makro OOF": round(score, 4)})
print(pd.DataFrame(rows).to_string(index=False))

## Simpan OOF
Format bersama (dibaca `ensemble_oof`): `oof_<MODEL_NAME>.npy` dan `test_<MODEL_NAME>.npy`,
float32, n × jumlah kelas, urutan kelas = `np.unique(train[TARGET])`. Satu baris per run ke
`percobaan.csv`.

In [ ]:
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof.astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_pred.astype("float32"))

log_path = f"{OOF_FOLDER}/percobaan.csv"
note = f"{len(NUM_COLS)} angka, {len(CAT_COLS)} kategori, {len(TEXT_COLS)} teks, svd {SVD_DIM}"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "teks_tabular",
                         "model": MODEL_NAME, "metrik": "f1_makro", "skor_cv": round(SCORE, 4),
                         "detik": SECONDS, "catatan": note}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(3).to_string(index=False))

## Prediksi test + submission
Kelas = `argmax` dari rata-rata peluang model per lipatan (tidak latih ulang di semua data:
format bersama). Id dan urutan baris diambil dari sample_submission.

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_labels = CLASSES[test_pred.argmax(axis=1)]
pred_by_id = pd.Series(pred_labels, index=test[ID_COL].values)

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
print(submission.shape)
print(submission[SUB_COL].value_counts().sort_index())
submission.head()

## Cek submission
Dua sel: struktur, lalu isi. Hanya mencetak. Hasil akhir harus `MASALAH: 0`.

In [ ]:
def check_structure(written, sample):
    problems = []
    if list(written.columns) != list(sample.columns):
        problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
    if len(written) != len(sample):
        problems.append(f"jumlah baris {len(written)} != sample {len(sample)}")
    if written[ID_COL].duplicated().sum() > 0:
        problems.append("id kembar")
    if set(written[ID_COL]) != set(sample[ID_COL]):
        problems.append("himpunan id beda dari sample")
    elif list(written[ID_COL]) != list(sample[ID_COL]):
        problems.append("urutan id beda dari sample")
    return problems


written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = check_structure(written, sample)

In [ ]:
ALLOWED = CLASSES.tolist()   # label sah = label train

for col in sample.columns:
    if col in written.columns:
        missing = written[col].isna()
        if missing.sum() > 0:
            problems.append(f"{col}: {missing.sum()} nilai kosong")
        if written[col].dtype != sample[col].dtype:
            problems.append(f"{col}: tipe {written[col].dtype}, sample {sample[col].dtype}")
foreign = set(written[SUB_COL].dropna()) - set(ALLOWED)
if len(foreign) > 0:
    problems.append(f"{SUB_COL}: label di luar ALLOWED {sorted(foreign)[:5]}")
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = submission[ID_COL].map(key_by_id)
test_f1 = f1_score(truth, submission[SUB_COL], average="macro")
print("F1 makro test:", round(test_f1, 4), "| F1 makro OOF:", round(SCORE, 4),
      "| selisih:", round(SCORE - test_f1, 4))
print("akurasi test :", round(accuracy_score(truth, submission[SUB_COL]), 4))